In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
import datetime
from torch.utils.data import DataLoader
from tqdm import tqdm


from model import (
    EnhancedFinGAT, 
    FinGATDataset, 
    custom_collate_fn,
    create_time_period_dataloaders
)

def print_metrics_table(metrics, description="Test Set", momentum_weight=0.80):
    """
    Print metrics in a formatted table
    
    Args:
        metrics: Dictionary of metrics returned from evaluate_model_performance
        description: Description of the dataset (e.g., "Test Set")

    """
    # Extract metrics
    mae = metrics['mae']
    acc = metrics['movement_accuracy']
    
    # Print header
    print("\n" + "=" * 80)
    print("-" * 80)
    print(f"{'':15} | {'MRR':10} | {'Precision':10} | {'IRR':10} | {'MAE':10} | {'Acc':10}")
    print("-" * 80)
    
    # Print metrics for each portfolio size
    for top_n in sorted([int(k.split('_')[1]) for k in metrics['portfolio_metrics'].keys()]):
        portfolio_key = f'top_{top_n}'
        if portfolio_key in metrics['portfolio_metrics']:
            portfolio = metrics['portfolio_metrics'][portfolio_key]
            precision = portfolio['ranking_precision']
            irr = portfolio['cumulative_return']  # Non-annualized IRR
            mrr = portfolio['mrr']  # Get top_n specific MRR
            
            print(f"K={top_n:<14} | {mrr:<10.4f} | {precision:<10.4f} | {irr:<10.4f} | {mae:<10.6f} | {acc:<10.4f}")
    
    print("=" * 80)

def evaluate_model_performance(model, data_loader, device, company_map=None, top_ns=[5, 10, 20], save_daily_csv=False, output_dir=None):
    """
    Evaluate model performance with comprehensive metrics including MAE
    
    Args:
        model: Trained model
        data_loader: Validation/test data loader
        device: Computation device
        company_map: Dictionary mapping company indices to actual company tickers
        top_ns: List of portfolio sizes to evaluate
        save_daily_csv: Whether to save daily predictions to CSV files
        output_dir: Directory to save CSV files if save_daily_csv is True
        
    Returns:
        Dictionary containing all calculated metrics
    """
    model.eval()
    all_predictions = []
    all_returns = []
    all_dates = []
    all_tickers = []
    all_company_indices = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating model"):
            features, sector_indices, company_indices, dates, returns, movements = batch
            
            features = features.to(device)
            sector_indices = sector_indices.to(device)
            
            # Get model predictions
            return_preds, movement_preds = model(features, sector_indices)
            
            # Store predictions, returns, dates, and company indices
            all_predictions.extend(return_preds.cpu().numpy())
            all_returns.extend(returns.cpu().numpy())
            all_dates.extend(dates)
            all_company_indices.extend(company_indices.cpu().numpy())
            
            # Store generic tickers for now
            all_tickers.extend([f"company_{idx}" for idx in company_indices.cpu().numpy()])
    
    # Calculate MAE (Mean Absolute Error)
    mae = np.mean(np.abs(np.array(all_predictions) - np.array(all_returns)))
    
    # Create DataFrame with predictions, returns, dates, and tickers
    results_df = pd.DataFrame({
        'prediction': all_predictions,
        'actual_return': all_returns,
        'date': all_dates,
        'ticker': all_tickers,
        'company_idx': all_company_indices
    })
    
    # Calculate movement accuracy (sign prediction)
    movement_correct = ((results_df['prediction'] > 0) == (results_df['actual_return'] > 0)).mean()
    
    # Save daily CSV files if requested
    if save_daily_csv and output_dir:
        os.makedirs(output_dir, exist_ok=True)
        
        # Group by date and save each day's predictions and actual returns
        for date, group in results_df.groupby('date'):
            date_str = date.strftime('%Y-%m-%d')
            filename = os.path.join(output_dir, f"predictions_{date_str}.csv")
            
            # Create rank columns for both predicted and actual returns
            # Sort by actual returns (descending) to get actual rankings
            actual_sorted = group.sort_values('actual_return', ascending=False).copy()
            actual_sorted['actual_rank'] = range(1, len(actual_sorted) + 1)
            
            # Create mapping from index to actual rank
            actual_rank_mapping = actual_sorted['actual_rank'].to_dict()
            
            # Sort by prediction (descending) to get predicted rankings
            sorted_group = group.sort_values('prediction', ascending=False).copy()
            sorted_group['predicted_rank'] = range(1, len(sorted_group) + 1)
            
            # Add actual ranks to the prediction-sorted dataframe
            sorted_group['actual_rank'] = sorted_group.index.map(actual_rank_mapping)
            
            # Map company indices to real ticker names if company_map is provided
            if company_map is not None:
                sorted_group['real_ticker'] = sorted_group['company_idx'].map(
                    lambda idx: company_map.get(int(idx), f"company_{int(idx)}")
                )
                # Keep important columns and rename
                sorted_group = sorted_group[['real_ticker', 'prediction', 'actual_return', 'predicted_rank', 'actual_rank']]
                sorted_group.columns = ['ticker', 'prediction', 'actual_return', 'predicted_rank', 'actual_rank']
            else:
                # Keep the generic ticker and important columns
                sorted_group = sorted_group[['ticker', 'prediction', 'actual_return', 'predicted_rank', 'actual_rank']]
            
            # Add a column indicating if this stock is in top N for both predictions and actuals
            for top_n in top_ns:
                if len(sorted_group) >= top_n:
                    sorted_group[f'in_top_{top_n}_predicted'] = sorted_group['predicted_rank'] <= top_n
                    sorted_group[f'in_top_{top_n}_actual'] = sorted_group['actual_rank'] <= top_n
            
            # Save to CSV
            sorted_group.to_csv(filename, index=False)
            print(f"Saved predictions and rankings for {date_str} to {filename}")
    
    
    mrr_by_topn = {top_n: [] for top_n in top_ns}
    
    for date, group in results_df.groupby('date'):
        num_stocks = len(group)
        
        # Sort by actual returns (descending)
        actual_sorted = group.sort_values('actual_return', ascending=False)
        # Assign rank based on actual returns (1-based)
        actual_sorted['actual_rank'] = range(1, len(actual_sorted) + 1)
        
        # Create a mapping from index to actual rank
        actual_rank_mapping = actual_sorted['actual_rank'].to_dict()
        
        # Sort by predictions (descending)
        pred_sorted = group.sort_values('prediction', ascending=False)
        # Assign rank based on predictions (1-based)
        pred_sorted['pred_rank'] = range(1, len(pred_sorted) + 1)
        
        # Add actual ranks to prediction-sorted dataframe
        pred_sorted['actual_rank'] = pred_sorted.index.map(actual_rank_mapping)
        
        # Calculate MRR for each top_n
        for top_n in top_ns:
            if len(pred_sorted) >= top_n:
                # Get top N predicted stocks
                pred_top_n = pred_sorted.nsmallest(top_n, 'pred_rank')
                
                # Calculate reciprocal rank based on actual ranks of predicted top stocks
                reciprocal_ranks = [1.0 / rank if rank <= top_n else 0 
                                   for rank in pred_top_n['actual_rank']]
                date_mrr = np.mean(reciprocal_ranks)
                mrr_by_topn[top_n].append(date_mrr)
    
    # Calculate overall MRR for each top_n
    mrr_dict = {}
    for top_n in top_ns:
        mrr = np.mean(mrr_by_topn[top_n]) if mrr_by_topn[top_n] else 0
        mrr_dict[top_n] = mrr
    
    # Calculate IRR and ranking precision for each top_n
    portfolio_metrics = {}
    for top_n in top_ns:
        daily_returns = []
        precision_values = []
        
        for date, group in results_df.groupby('date'):
            if len(group) < top_n:
                continue
                
            # Sort by actual and predicted returns
            actual_top = set(group.nlargest(top_n, 'actual_return').index)
            pred_top = set(group.nlargest(top_n, 'prediction').index)
            
            # Calculate precision (what fraction of predicted top_n were actually top_n)
            precision = len(actual_top.intersection(pred_top)) / top_n
            precision_values.append(precision)
            
            # Calculate portfolio return using equal weighting
            pred_portfolio = group.nlargest(top_n, 'prediction')
            portfolio_return = pred_portfolio['actual_return'].mean()
            daily_returns.append(portfolio_return)
        
        # Calculate cumulative return
        cumulative_return = np.prod(1 + np.array(daily_returns)) - 1
        
        # Calculate annualized IRR (assuming daily data)
        trading_days_per_year = 252
        n_days = len(daily_returns)
        if n_days > 0:
            annualized_irr = (1 + cumulative_return) ** (trading_days_per_year / n_days) - 1
        else:
            annualized_irr = 0
            
        # Average precision across all dates
        avg_precision = np.mean(precision_values) if precision_values else 0
        
        portfolio_metrics[f'top_{top_n}'] = {
            'cumulative_return': cumulative_return,
            'annualized_irr': annualized_irr,
            'ranking_precision': avg_precision,
            'daily_returns': daily_returns,
            'mrr': mrr_dict[top_n]  # Add the top_n specific MRR
        }
    
    # Compile all metrics
    all_metrics = {
        'movement_accuracy': movement_correct,
        'mrr': mrr_dict[top_ns[0]],  # Use the first top_n as overall MRR for backward compatibility
        'mae': mae,
        'portfolio_metrics': portfolio_metrics
    }
    
    # Return all metrics for use in printing the table
    return all_metrics

def main():
    # Setup device
    if torch.backends.mps.is_available():
        print("MPS backend is available!")
        device = torch.device("mps")
    else:
        print("MPS backend is not available. Using CUDA if available, otherwise CPU.")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Using device: {device}")
    
    # Load data
    print("Loading data...")
    data_path = 'stock_data/processed/merged_stock_data_with_enhanced_features___.parquet'
    
    if not os.path.exists(data_path):
        print(f"Error: Data file {data_path} not found.")
        print("Please specify the correct path to your stock data.")
        return
    
    df = pd.read_parquet(data_path)
    
    # Make sure index is datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    
    # Define parameters (same as in your training)
    hidden_dim = 128
    embed_dim = 32
    nhead = 8
    num_sectors = 19
    num_layers = 2
    learning_rate = 1e-5
    weight_decay = 0.001
    alpha = 0.7
    batch_size = 64
    sequence_length = 15
    # Define test period
    test_start_date = "2025-01-11"
    test_end_date = "2025-03-22"
    
    # Create test dataloader
    test_loader = create_time_period_dataloaders(
        df, batch_size, sequence_length, test_start_date, test_end_date
    )
    
    # Check if we have data
    if test_loader is None:
        print("Error: Test dataloader could not be created. Check your date ranges.")
        return
    
    # Create a reverse mapping from company indices to real company names
    test_dataset = test_loader.dataset
    company_map = {}
    for company_name, idx in test_dataset.company_map.items():
        company_map[idx] = company_name
    
    print(f"Created mapping for {len(company_map)} company tickers")
    

    sample_batch = next(iter(test_loader))
    features, _, _, _, _, _ = sample_batch
    input_dim = features.shape[2]
    print(f"Input dimension: {input_dim}")
    

    feature_cols = [col for col in df.columns.levels[2] if col != 'return_ratio']
    technical_indicator = feature_cols.index('adv_momentum')
    print(f"Indicator feature found at index: {technical_indicator}")
    
    # Load the trained model
    model_path = 'fingat_model_best.pt'  
    if not os.path.exists(model_path):
        print(f"Error: Model file {model_path} not found.")
        return
    
    # Create the model with the same architecture
    model = EnhancedFinGAT(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        embed_dim=embed_dim,
        num_sectors=num_sectors,
        technical_indicator=technical_indicator 
    )
    
    # Load the trained weights
    print(f"Loading model from {model_path}...")
    try:
        # Add weights_only=False to allow loading NumPy objects from the saved model
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model = model.to(device)
        print(f"Model successfully loaded from {model_path}")
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        return

    output_dir = "output_results"
    
    # Evaluate on test set and save daily predictions
    print("\n======== TEST SET EVALUATION (Jan 11, 2025 - March 22, 2025) ========")
    test_metrics = evaluate_model_performance(
        model, 
        test_loader, 
        device,
        company_map=company_map,  # Pass the company name mapping
        top_ns=[5, 10, 20],
        save_daily_csv=True,
        output_dir=output_dir
    )
    
    # Print metrics table
    print_metrics_table(test_metrics, "Test Set", momentum_weight=0.20)
    
    # Save overall metrics to JSON for reference
    import json
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        # Convert numpy values to Python native types for JSON serialization
        metrics_json = {}
        metrics_json['movement_accuracy'] = float(test_metrics['movement_accuracy'])
        metrics_json['mae'] = float(test_metrics['mae'])
        metrics_json['portfolio_metrics'] = {}
        
        for key, value in test_metrics['portfolio_metrics'].items():
            metrics_json['portfolio_metrics'][key] = {
                'cumulative_return': float(value['cumulative_return']),
                'annualized_irr': float(value['annualized_irr']),
                'ranking_precision': float(value['ranking_precision']),
                'mrr': float(value['mrr'])
            }
        
        json.dump(metrics_json, f, indent=4)
    
    print(f"\nEvaluation complete! Daily prediction CSVs saved to {output_dir}/")

if __name__ == "__main__":
    main()

[W502 12:13:58.536931000 init.cpp:858] Warning: Use _jit_set_fusion_strategy, bailout depth is deprecated. Setting to (STATIC, 2) (function operator())


MPS backend is available!
Using device: mps
MPS backend is available!
Using device: mps
Loading data...
Creating dataloader for period: 2025-01-11 to 2025-03-22
Number of trading days in period: 49
Created mapping for 445 company tickers
Input dimension: 25
Indicator feature found at index: 13
Model config: input_dim=25, hidden_dim=128, embed_dim=32
Loading model from fingat_model_best.pt...
Model successfully loaded from fingat_model_best.pt

======== TEST SET EVALUATION (Jan 11, 2025 - March 22, 2025) ========


Evaluating model: 100%|██████████| 237/237 [00:14<00:00, 16.04it/s]


Saved predictions and rankings for 2025-02-01 to output_results/predictions_2025-02-01.csv
Saved predictions and rankings for 2025-02-03 to output_results/predictions_2025-02-03.csv
Saved predictions and rankings for 2025-02-04 to output_results/predictions_2025-02-04.csv
Saved predictions and rankings for 2025-02-05 to output_results/predictions_2025-02-05.csv
Saved predictions and rankings for 2025-02-06 to output_results/predictions_2025-02-06.csv
Saved predictions and rankings for 2025-02-07 to output_results/predictions_2025-02-07.csv
Saved predictions and rankings for 2025-02-10 to output_results/predictions_2025-02-10.csv
Saved predictions and rankings for 2025-02-11 to output_results/predictions_2025-02-11.csv
Saved predictions and rankings for 2025-02-12 to output_results/predictions_2025-02-12.csv
Saved predictions and rankings for 2025-02-13 to output_results/predictions_2025-02-13.csv
Saved predictions and rankings for 2025-02-14 to output_results/predictions_2025-02-14.csv